## Import libraries


In [16]:
import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchcrop
from torchcrop import (
    CropParameters,
    SoilParameters,
    SiteParameters,
    Lintul5Model,
    WeatherDriver
)

DATA_DIR = Path("data", "brandenburg", "torchcrop")

In [ ]:
class TorchCropDataset(Dataset):
    def __init__(self, weather_dir, soil_dir, site_dir):
        super().__init__()
        self.weather_dir = weather_dir
        self.soil_data = pd.read_csv(os.path.join(soil_dir, "soil.csv")).set_index("location")
        # self.site_data = pd.read_csv(os.path.join(site_dir, "site.csv"))
        self.locations = self.soil_data.index
        
    def __len__(self):
        return len(self.locations)
        
    def __getitem__(self, idx):
        
        # Prepare weather data
        # ----------------------------------------------------------------------------------
        weather_data = pd.read_csv(os.path.join(self.weather_dir, f"{idx}.csv"), parse_dates=["Date"])
        weather_data["Radiation"] = weather_data["Radiation"] / 1000.0      # Convert radiation unit from kJ to MJ
        weather_data["Date"] = weather_data["Date"].dt.dayofyear            # Convert date to DOY
        weather_data = weather_data.values                                  # Final weather array

        # Prepare soil data
        # ----------------------------------------------------------------------------------
        soil_over = dict(
            wcad=sc(srow["SMDRY"]),     # air-dry water content
            wcwp=sc(srow["SMW"]),       # wilting point
            wcfc=sc(srow["SMFC"]),      # field capacity
            wcst=sc(srow["SM0"]),       # saturation
            crairc=sc(srow["CRAIRC"]),  # critical air content
            wci=sc(srow["SMI"]),        # initial root-zone moisture
            wci_lower=sc(srow["SMLOWI"]),
            rdmso=sc(srow["RDMSO"]),    # max rooting depth from soil [m]
            runfr=sc(srow["RUNFR"]),
            cfev=sc(srow["CFEV"]),
            ksub=sc(srow["KSUB"]),
            irri=sc(CFG["irri"]),       # no irrigation
        )
        
        return weather_data
        

In [83]:
dataset = TorchCropDataset(
    weather_dir=DATA_DIR / "weather",
    soil_dir=DATA_DIR / "soil",
    site_dir=DATA_DIR / "site"
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
)

In [87]:
dataset.soil_data

,SMDRY,SMW,SMFC,SMO,CRAIRC,SMI,SMLOWI,RDMSO,RUNFR,CFEV,KSUB
location,,,,,,,,,,,
0,0.026275,0.052550,0.132392,0.375575,0.07,0.132392,0.132392,2,0,2,100
1,0.028806,0.057611,0.148077,0.391740,0.07,0.148077,0.148077,2,0,2,100
2,0.061266,0.122533,0.239435,0.416913,0.07,0.239435,0.239435,2,0,2,100
3,0.016730,0.033460,0.079727,0.372395,0.07,0.079727,0.079727,2,0,2,100
4,0.039435,0.078871,0.178746,0.364946,0.07,0.178746,0.178746,2,0,2,100
5,0.041183,0.082366,0.154016,0.404326,0.07,0.154016,0.154016,2,0,2,100
6,0.020010,0.040021,0.100029,0.335405,0.07,0.100029,0.100029,2,0,2,100
7,0.027342,0.054683,0.128910,0.385821,0.07,0.128910,0.128910,2,0,2,100
8,0.021794,0.043587,0.096162,0.352463,0.07,0.096162,0.096162,2,0,2,100


In [ ]:
dataset.soil_data.set_index("location").iloc

SMDRY       0.028806
SMW         0.057611
SMFC        0.148077
SMO         0.391740
CRAIRC      0.070000
SMI         0.148077
SMLOWI      0.148077
RDMSO       2.000000
RUNFR       0.000000
CFEV        2.000000
KSUB      100.000000
Name: 1, dtype: float64

In [73]:
batch = next(iter(dataloader))

In [74]:
WeatherDriver(batch).data[0][:, 0]

tensor([  1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,  11.,  12.,
         13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,  22.,  23.,  24.,
         25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,  33.,  34.,  35.,  36.,
         37.,  38.,  39.,  40.,  41.,  42.,  43.,  44.,  45.,  46.,  47.,  48.,
         49.,  50.,  51.,  52.,  53.,  54.,  55.,  56.,  57.,  58.,  59.,  60.,
         61.,  62.,  63.,  64.,  65.,  66.,  67.,  68.,  69.,  70.,  71.,  72.,
         73.,  74.,  75.,  76.,  77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,
         85.,  86.,  87.,  88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,
         97.,  98.,  99., 100., 101., 102., 103., 104., 105., 106., 107., 108.,
        109., 110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
        121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131., 132.,
        133., 134., 135., 136., 137., 138., 139., 140., 141., 142., 143., 144.,
        145., 146., 147., 148., 149., 15

In [ ]:
weather.data[0][:, 0]

In [62]:
batch.shape

torch.Size([8, 730, 8])

In [55]:
help(DataLoader)

Help on class DataLoader in module torch.utils.data.dataloader:

class DataLoader(typing.Generic)
 |  DataLoader(dataset: 'Dataset[_T_co]', batch_size: 'int | None' = 1, shuffle: 'bool | None' = None, sampler: 'Sampler | Iterable | None' = None, batch_sampler: 'Sampler[list] | Iterable[list] | None' = None, num_workers: 'int' = 0, collate_fn: '_collate_fn_t | None' = None, pin_memory: 'bool' = False, drop_last: 'bool' = False, timeout: 'float' = 0, worker_init_fn: '_worker_init_fn_t | None' = None, multiprocessing_context=None, generator=None, *, prefetch_factor: 'int | None' = None, persistent_workers: 'bool' = False, pin_memory_device: 'str' = '', in_order: 'bool' = True) -> 'None'
 |  
 |  Data loader combines a dataset and a sampler, and provides an iterable over the given dataset.
 |  
 |  The :class:`~torch.utils.data.DataLoader` supports both map-style and
 |  iterable-style datasets with single- or multi-process loading, customizing
 |  loading order and optional automatic batc

In [ ]:
dataset.soil_data.loc[]

,location,SMDRY,SMW,SMFC,SMO,CRAIRC,SMI,SMLOWI,RDMSO,RUNFR,CFEV,KSUB
0,0,0.026275,0.052550,0.132392,0.375575,0.07,0.132392,0.132392,2,0,2,100
1,1,0.028806,0.057611,0.148077,0.391740,0.07,0.148077,0.148077,2,0,2,100
2,2,0.061266,0.122533,0.239435,0.416913,0.07,0.239435,0.239435,2,0,2,100
3,3,0.016730,0.033460,0.079727,0.372395,0.07,0.079727,0.079727,2,0,2,100
4,4,0.039435,0.078871,0.178746,0.364946,0.07,0.178746,0.178746,2,0,2,100
5,5,0.041183,0.082366,0.154016,0.404326,0.07,0.154016,0.154016,2,0,2,100
6,6,0.020010,0.040021,0.100029,0.335405,0.07,0.100029,0.100029,2,0,2,100
7,7,0.027342,0.054683,0.128910,0.385821,0.07,0.128910,0.128910,2,0,2,100
8,8,0.021794,0.043587,0.096162,0.352463,0.07,0.096162,0.096162,2,0,2,100
9,9,0.029174,0.058347,0.122051,0.322010,0.07,0.122051,0.122051,2,0,2,100


In [51]:
dataset[1]

array([[ 1.00e+00,  2.10e+00, -2.10e+00, ...,  0.00e+00,  5.10e-01,
         1.80e+00],
       [ 2.00e+00,  9.00e-01, -2.70e+00, ...,  0.00e+00,  5.00e-01,
         2.20e+00],
       [ 3.00e+00,  3.90e+00,  8.00e-01, ...,  4.00e+00,  6.40e-01,
         3.60e+00],
       ...,
       [ 3.63e+02,  1.70e+00,  1.00e-01, ...,  0.00e+00,  6.00e-01,
         2.50e+00],
       [ 3.64e+02,  7.10e+00,  9.00e-01, ...,  1.80e+00,  6.60e-01,
         3.50e+00],
       [ 3.65e+02,  1.23e+01,  1.10e+01, ...,  1.60e+00,  1.26e+00,
         3.80e+00]], shape=(730, 8))

In [52]:
len(dataset)

18